In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from Logic_game import CaroLogic
import math
import time
import concurrent.futures

depth_a = 3
depth_b = 3
NUM_GAMES_SIMULATE = 150

ATTACK_PATTERNS = [
    (1, 1, 1, 1, 1),       # index 0: Ăn 5
    (0, 1, 1, 1, 1, 0),    # index 1: 4 thoáng
    (0, 1, 1, 1, 1),       # index 2: 4 chặn 1 đầu
    (1, 1, 1, 1, 0),       # index 3: 4 chặn 1 đầu
    (0, 1, 1, 1, 0),       # index 4: 3 thoáng
    (0, 1, 1, 0, 1, 0),    # index 5: 3 thoáng có lỗ
    (0, 1, 0, 1, 1, 0),    # index 6: 3 thoáng có lỗ
    (0, 1, 1, 1),          # index 7: 3 chặn
    (1, 1, 1, 0),          # index 8: 3 chặn
    (0, 1, 1, 0)           # index 9: 2 thoáng
]

DEFEND_PATTERNS = [
    (-1, -1, -1, -1, -1),   # index 10
    (0, -1, -1, -1, -1, 0), # index 11
    (0, -1, -1, -1, -1),    # index 12
    (-1, -1, -1, -1, 0),    # index 13
    (0, -1, -1, -1, 0),     # index 14
    (0, -1, -1, 0, -1, 0),  # index 15
    (0, -1, 0, -1, -1, 0),  # index 16
    (0, -1, -1, -1),        # index 17
    (-1, -1, -1, 0),        # index 18
    (0, -1, -1, 0)          # index 19
]

def count_pattern_occurrences(board, pattern):
    count = 0
    size = len(board)
    p_len = len(pattern)

    for r in range(size):
        for c in range(size):
            if c <= size - p_len:
                if tuple([board[r][c+i] for i in range(p_len)]) == pattern: count += 1
            if r <= size - p_len:
                if tuple([board[r+i][c] for i in range(p_len)]) == pattern: count += 1
            if r <= size - p_len and c <= size - p_len:
                if tuple([board[r+i][c+i] for i in range(p_len)]) == pattern: count += 1
            if r >= p_len - 1 and c <= size - p_len:
                if tuple([board[r-i][c+i] for i in range(p_len)]) == pattern: count += 1
    return count

def extract_features(board, player):
    features = [0] * 20
    normalized_board = [
        [board[r][c] if player == 1 else -board[r][c] for c in range(len(board))]
        for r in range(len(board))
    ]

    for idx, pattern in enumerate(ATTACK_PATTERNS):
        features[idx] = count_pattern_occurrences(normalized_board, pattern)
    for idx, pattern in enumerate(DEFEND_PATTERNS):
        features[10 + idx] = count_pattern_occurrences(normalized_board, pattern)

    return features

def play_one_game(game_id):
    logic = CaroLogic(board_size=10, bot_depth=3)
    logic.reset_game()
    game_active = True
    turn = 1

    X_history = []
    player_history = []
    winner = 0

    while game_active:
        if turn == 1:
            _, move = logic.minimax(logic.board, depth=depth_a, alpha=-math.inf, beta=math.inf, maximizing_player=True)
        else:
            _, move = logic.minimax(logic.board, depth=depth_b, alpha=-math.inf, beta=math.inf, maximizing_player=False)

        if not move:
            break # Bàn cờ hòa hoặc lỗi

        r, c = move
        success, is_win, _ = logic.play_move(r, c, turn)

        if success:
            features = extract_features(logic.board, turn)
            X_history.append(features)
            player_history.append(turn)

        if is_win:
            winner = turn
            game_active = False
        else:
            turn = -1 if turn == 1 else 1


    X_local = []
    y_local = []
    for i in range(len(X_history)):
        X_local.append(X_history[i])
        # Nếu ván đấu có người thắng, toàn bộ nước đi của người đó được tính là tốt (y=1)
        # Các nước đi của kẻ thua, hoặc nếu ván hòa (winner=0) đều bị gán là xấu/thường (y=0)
        if winner != 0 and player_history[i] == winner:
            y_local.append(1)
        else:
            y_local.append(0)

    print(f"-> [CPU Worker] Đã giả lập xong ván {game_id}/{NUM_GAMES_SIMULATE} (Kết quả: Phe {'1' if winner==1 else ('-1' if winner==-1 else 'Hòa')} thắng).")
    return X_local, y_local

def generate_dataset_parallel():
    print(f"Bắt đầu khởi chạy {NUM_GAMES_SIMULATE} ván cờ tự đấu (CHẾ ĐỘ ĐA NHÂN CPU CỰC HẠN)...")
    X_data, y_data = [], []

    with concurrent.futures.ProcessPoolExecutor() as executor:
        results = executor.map(play_one_game, range(1, NUM_GAMES_SIMULATE + 1))
        for X_local, y_local in results:
            X_data.extend(X_local)
            y_data.extend(y_local)

    return np.array(X_data), np.array(y_data)

def train_logistic_regression():
    X, y = generate_dataset_parallel()

    if len(np.unique(y)) < 2:
        print("LỖI HỆ THỐNG: Tập dữ liệu sinh ra không có ván thắng nào, không thể phân lớp.")
        return

    print(f"\n[Dữ liệu] Tổng số nước đi thu thập: {X.shape[0]} nước.")
    print(f"[Dữ liệu] Số lượng nước đi thuộc chiến thuật thắng (y=1): {np.sum(y)} nước.")

    # Chia tập dữ liệu (80% Train, 20% Test)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    print(f"[Dữ liệu] Đã chia tập Train: {X_train.shape[0]} mẫu | Tập Test: {X_test.shape[0]} mẫu.")
    print("Đang tiến hành huấn luyện mô hình Hồi quy Logistic trên tập Train...")

    # Khởi tạo mô hình Học máy
    model = LogisticRegression(
        solver='lbfgs',
        max_iter=2000,
        fit_intercept=True,
        class_weight='balanced',
        random_state=42
    )
    model.fit(X_train, y_train)

    # Đánh giá thuật toán trên tập Test độc lập
    y_pred_test = model.predict(X_test)

    print("\n" + "="*60)
    print(" BÁO CÁO ĐÁNH GIÁ CHẤT LƯỢNG MÔ HÌNH TRÊN TẬP TEST ")
    print("="*60)
    print(classification_report(y_test, y_pred_test, target_names=['Nước của phe Thua/Hòa (0)', 'Nước của phe Thắng (1)']))

    f1_win = f1_score(y_test, y_pred_test)
    print(f"=> Chỉ số F1-Score (Độ chính xác chiến thuật thực chất) đạt: {f1_win:.4f}")
    print("="*60)

    # Xuất ma trận Heuristic
    raw_weights = model.coef_[0]

    # Chuẩn hóa về hệ điểm lớn (chỉ chuẩn hóa các thế từ index 1 trở đi, giữ ăn 5 ở mức vô cực)
    max_val = np.max(np.abs(raw_weights[1:])) if np.max(np.abs(raw_weights[1:])) > 0 else 1.0
    scaled_weights = (np.abs(raw_weights) / max_val) * 80000

    print("\n" + "="*60)
    print(" BỘ TRỌNG SỐ HEURISTIC ĐÃ KHẮC PHỤC LỖI LOGIC TOÁN HỌC ")
    print("="*60)

    print("--- CẬP NHẬT CHO ATTACK_MATRIX (Trọng số dương) ---")
    print(f"  {ATTACK_PATTERNS[0]}: 200000,") # Gán chết điểm ăn 5
    for i in range(1, len(ATTACK_PATTERNS)):
        val = max(10.0, scaled_weights[i])
        print(f"  {ATTACK_PATTERNS[i]}: {val:.2f},")

    print("\n--- CẬP NHẬT CHO DEFEND_MATRIX (Trọng số âm) ---")
    print(f"  {DEFEND_PATTERNS[0]}: -200000,") # Gán chết điểm chặn 5
    for i in range(1, len(DEFEND_PATTERNS)):
        val = max(10.0, scaled_weights[10 + i])
        if i == 1: # Nhấn mạnh phạt nặng thế 4 thoáng của địch
            val = val * 2.5
        print(f"  {DEFEND_PATTERNS[i]}: {-val:.2f},")
    print("="*60)

if __name__ == "__main__":
    start_time = time.time()
    train_logistic_regression()
    print(f"\nThời gian chạy tổng hợp: {time.time() - start_time:.2f} giây")

Bắt đầu khởi chạy 150 ván cờ tự đấu (CHẾ ĐỘ ĐA NHÂN CPU CỰC HẠN)...
-> [CPU Worker] Đã giả lập xong ván 1/150 (Kết quả: Phe 1 thắng).
-> [CPU Worker] Đã giả lập xong ván 2/150 (Kết quả: Phe 1 thắng).
-> [CPU Worker] Đã giả lập xong ván 3/150 (Kết quả: Phe 1 thắng).
-> [CPU Worker] Đã giả lập xong ván 4/150 (Kết quả: Phe 1 thắng).
-> [CPU Worker] Đã giả lập xong ván 5/150 (Kết quả: Phe 1 thắng).
-> [CPU Worker] Đã giả lập xong ván 6/150 (Kết quả: Phe 1 thắng).
-> [CPU Worker] Đã giả lập xong ván 8/150 (Kết quả: Phe 1 thắng).
-> [CPU Worker] Đã giả lập xong ván 7/150 (Kết quả: Phe 1 thắng).
-> [CPU Worker] Đã giả lập xong ván 9/150 (Kết quả: Phe 1 thắng).
-> [CPU Worker] Đã giả lập xong ván 10/150 (Kết quả: Phe 1 thắng).
-> [CPU Worker] Đã giả lập xong ván 11/150 (Kết quả: Phe 1 thắng).
-> [CPU Worker] Đã giả lập xong ván 12/150 (Kết quả: Phe 1 thắng).
-> [CPU Worker] Đã giả lập xong ván 13/150 (Kết quả: Phe 1 thắng).
-> [CPU Worker] Đã giả lập xong ván 14/150 (Kết quả: Phe 1 thắng).
-> 